# 🔐 Unity Catalog Secrets — End-to-End Demo

**Unity Catalog secrets** store sensitive values (passwords, tokens, API keys) as *securable objects* under the
three-level namespace `catalog.schema.secret_name`. Unlike legacy **workspace-scoped** secrets (organized into flat
"scopes"), UC secrets are governed by Unity Catalog privileges and are available across **every workspace** attached
to the metastore.

| Privilege | What it allows |
|---|---|
| `CREATE SECRET` | Create secrets in a schema |
| `READ SECRET` | Retrieve a secret's value |
| `WRITE SECRET` | Update / rotate a secret's value |
| `REFERENCE SECRET` | Let a UC object (e.g. a Connection) use the secret **without** exposing its value |

**Compute requirement:** Serverless **environment version ≥ 4**, or classic compute in **Standard access mode on DBR 17.3 LTS+**.


> 🛡️ Secret values are **automatically redacted** in notebook output and logs when read via `dbutils.secrets.get`.


In [0]:
import requests, json

# --- Target three-level namespace 
CATALOG = "your catalog"
SCHEMA  = "your schema"
SECRET  = "your secret"
FQN     = f"{CATALOG}.{SCHEMA}.{SECRET}"         

# --- Authenticate using the notebook's own context ---
ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST  = ctx.apiUrl().get()
TOKEN = ctx.apiToken().get()
HDRS  = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
BASE  = f"{HOST}/api/2.1/unity-catalog/secrets"


# --- Create the secret (POST /api/2.1/unity-catalog/secrets) ---
# Requires: USE CATALOG + (schema ownership OR CREATE SECRET + USE SCHEMA)
resp = requests.post(
    BASE,
    headers=HDRS,
    json={
        "catalog_name": CATALOG,
        "schema_name": SCHEMA,
        "name": SECRET,
        "value": "OAI API Key",
        "comment": "OAI Token",
    },
)

if resp.status_code in (200, 201):
    print(f"\u2705 Secret created: {FQN}")
    print(json.dumps(resp.json(), indent=2))
elif resp.status_code == 409:
    print(f"\u2139\ufe0f  Secret already exists: {FQN} \u2014 continuing.")
else:
    print(f"\u274c Error creating secret:")
    print(json.dumps(resp.json(), indent=2))

# Secrets can be created in the UI

![Screenshot 2026-09-02 at 15.58.33_1788357584949.png](./Screenshot 2026-09-02 at 15.58.33_1788357584949.png "Screenshot 2026-09-02 at 15.58.33_1788357584949.png")

In [0]:
all_secrets = dbutils.secrets.list(catalog=CATALOG, schema=SCHEMA)
for i in all_secrets:
  print(i.key)


In [0]:
# Metadata only — values are never returned by the list endpoint.
resp = requests.get(
    BASE,
    headers=HDRS,
    params={"catalog_name": CATALOG, "schema_name": SCHEMA},
)
secrets = resp.json().get("secrets", [])
print(f"📋 {len(secrets)} secret(s) in {CATALOG}.{SCHEMA}:\n")
for s in secrets:
    print(f"  • {s['full_name']:<45}  owner={s.get('effective_owner','?')}  comment={s.get('comment','')}")

In [0]:

from openai import OpenAI

api_key = dbutils.secrets.get(catalog=CATALOG, schema=SCHEMA, key=SECRET)
# Instantiating the OpenAI client with the provided API key
client = OpenAI(api_key=api_key)

# Calling the chat completions API method to generate a model response
completion = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "system", "content": "You are an AI Guru who enjoy OSS Technologies"},
        {"role": "user", "content": "Which is the strongest Cloud Agnostic Data and AI Platform ?"},
    ],
    max_completion_tokens=2000,  # reasoning models need headroom for chain-of-thought + reply
)

answer = completion.choices[0].message.content
print(f"✅ Model: {completion.model}")
print(f"📊 Tokens used: {completion.usage.total_tokens}")
print(f"\n🤖 Answer:\n{answer}")

In [0]:


# Grant READ SECRET on the demo secret to a single user or group
resp = requests.patch(
    f"{HOST}/api/2.1/unity-catalog/permissions/secret/{FQN}",
    headers=HDRS,
    json={"changes": [{"principal": "your email", "add": ["READ_SECRET"]}]},
)
print(f"Grant READ_SECRET → HTTP {resp.status_code}")
print(json.dumps(resp.json(), indent=2))



In [0]:
# Requires: WRITE SECRET.  update_mask=* is required by the API.
resp = requests.patch(
    f"{BASE}/{FQN}",
    headers=HDRS,
    params={"update_mask": "*"},
    json={"value": "rotated-value-456"},
)
print("Rotation result:")
print(json.dumps(resp.json(), indent=2))

# Prove the rotation took effect (dbutils re-reads the live value):
rotated = dbutils.secrets.get(catalog=CATALOG, schema=SCHEMA, key=SECRET)
print(f"\n✅ Post-rotation value length: {len(rotated)} chars "
      f"(was 22, now {len(rotated)} → changed = {len(rotated) != 22})")

In [0]:
# Schema-level inheritance: let a teammate create + read secrets in the schema
requests.patch(f"{HOST}/api/2.1/unity-catalog/permissions/schema/youssef_demo.OAITT",
               headers=HDRS, json={"changes": [{"principal": "youssef.mrini@databricks.com", "add": ["CREATE_SECRET","USE_SCHEMA"]}]})
print(f"Grant READ_SECRET → HTTP {resp.status_code}")
print(json.dumps(resp.json(), indent=2))


In [0]:
# Allows a user to reference a secret, for example from a Unity Catalog connection, without access to the value.
requests.patch(f"{HOST}/api/2.1/unity-catalog/permissions/secret/{FQN}",
     headers=HDRS, json={"changes": [{"principal": "youssef.mrini@databricks.com", "add": ["REFERENCE_SECRET"]}]})

print(f"Grant READ_SECRET → HTTP {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

In [0]:
# WRITE SECRET allows the grantee to update / rotate the secret value
requests.patch(f"{HOST}/api/2.1/unity-catalog/permissions/secret/{FQN}",
    headers=HDRS, json={"changes": [{"principal": "your email", "add": ["WRITE_SECRET"]}]})
    
print(f"Grant READ_SECRET → HTTP {resp.status_code}")
print(json.dumps(resp.json(), indent=2))


In [0]:
# Revoke a previously granted privilege from a principal
requests.patch(f"{HOST}/api/2.1/unity-catalog/permissions/secret/{FQN}",
     headers=HDRS, json={"changes": [{"principal": "your email", "remove": ["READ_METADATA ","READ_SECRET","WRITE_SECRET","REFERENCE_SECRET"]}]})

print(f"Grant READ_SECRET → HTTP {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

In [0]:
# Who has what on this secret, including privileges inherited from schema/catalog/metastore.
resp = requests.get(
    f"{HOST}/api/2.1/unity-catalog/effective-permissions/secret/{FQN}",
    headers=HDRS,
)
print(f"🔎 Effective permissions on {FQN}:\n")
for pa in resp.json().get("privilege_assignments", []):
    print(f"  principal: {pa['principal']}")
    for p in pa.get("privileges", []):
        src = f" (inherited from {p['inherited_from_type']}: {p['inherited_from_name']})" if p.get("inherited_from_type") else ""
        print(f"      - {p['privilege']}{src}")

In [0]:
%sql
-- Secret activity is logged to the UC audit system table.
SELECT event_time, user_identity.email AS user, action_name, request_params
FROM system.access.audit
WHERE service_name = 'unityCatalog'
  AND action_name ILIKE '%Secret%'
  AND event_date >= current_date() - INTERVAL 1 DAY
ORDER BY event_time DESC
LIMIT 50;

In [0]:
# Requires: secret ownership or metastore admin.
resp = requests.delete(f"{BASE}/{FQN}", headers=HDRS)
if resp.status_code == 200:
    print(f"🧹 Deleted: {FQN}")
else:
    print(f"❌ Delete failed:")
    print(json.dumps(resp.json(), indent=2))

# Verify it's gone (expect a 'does not exist' error):
check = requests.get(f"{BASE}/{FQN}", headers=HDRS, params={"include_value": "true"})
print(f"\nVerification → HTTP {check.status_code}: "
      + ("still present ❗" if check.status_code == 200 else "confirmed removed ✅"))